In [1]:
!nvidia-smi

Fri Mar 13 16:18:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!apt-get update
!apt-get install -y cuda-toolkit-13-0

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [3]:
import os
os.environ["PATH"] = "/usr/local/cuda-13.0/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-13.0/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

In [4]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [5]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp7uutpx1d".


In [6]:
%%cuda
#include <cuda_runtime.h>
#include <iostream>

__global__ void add_arrays(
    const float *inA,
    const float *inB,
    float *result,
    int n)
{
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < n)
        result[id] = inA[id] + inB[id];
}

int main()
{
    const int N = 16;
    const int size = N * sizeof(float);

    // Allocate host memory
    float h_inA[N], h_inB[N], h_result[N];
    for (int i = 0; i < N; i++)
    {
        h_inA[i] = i * 1.0f;
        h_inB[i] = (N - i) * 1.0f;
    }

    // Allocate device memory
    float *d_inA, *d_inB, *d_result;
    cudaMalloc((void **)&d_inA, size);
    cudaMalloc((void **)&d_inB, size);
    cudaMalloc((void **)&d_result, size);

    // Copy inputs to device
    cudaMemcpy(d_inA, h_inA, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_inB, h_inB, size, cudaMemcpyHostToDevice);

    // Launch kernel
    int threadsPerBlock = 256;
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    add_arrays<<<blocks, threadsPerBlock>>>(d_inA, d_inB, d_result, N);
    cudaDeviceSynchronize();

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
    {
        std::cerr << "CUDA error: " << cudaGetErrorString(err) << std::endl;
        return 1;
    }

    // Copy result back
    cudaMemcpy(h_result, d_result, size, cudaMemcpyDeviceToHost);

    // Print
    for (int i = 0; i < N; i++)
    {
        std::cout << h_inA[i] << " + " << h_inB[i] << " = " << h_result[i] << std::endl;
    }

    // Free
    cudaFree(d_inA);
    cudaFree(d_inB);
    cudaFree(d_result);

    return 0;
}

0 + 16 = 16
1 + 15 = 16
2 + 14 = 16
3 + 13 = 16
4 + 12 = 16
5 + 11 = 16
6 + 10 = 16
7 + 9 = 16
8 + 8 = 16
9 + 7 = 16
10 + 6 = 16
11 + 5 = 16
12 + 4 = 16
13 + 3 = 16
14 + 2 = 16
15 + 1 = 16

